#### RAG 심화
- 중복 문서 문제: 비슷한 내용의 청크가 여러 개 반환되어 컨텍스트 낭비 발생
- 검색: 시멘틱 유사도 만으로는 정확한 키워드 매칭이 어려움
- 구조적 질의 불가: "2024년 이후 계약 금액이 1억 이상인 제품 찾기" 같은 메타 필터 처리 불가
- 노이즈 청크: 관련성이 낮은 청크가 LLM에게 전달되면 환각 유발

임배딩 바꿔보고, 청크 크기도 바꿔보고

In [ ]:
%pip install openai
%pip install langchain_openai
%pip install pypdf beautifulsoup4 youtube-transcript-api langchain-chroma faiss-cpu pdfplumber
%pip install langchain_text_splitters
%pip install chroma
%pip install langchain langchain-community langchain-ollama langchain-core python-dotenv
%pip install langchain-ibm ibm-watsonx-ai gradio
%pip install lark

In [44]:
# 라이브러리 로드
from langchain_ollama import ChatOllama
from langchain_ibm import ChatWatsonx

from langchain_ollama import OllamaEmbeddings
from langchain_ibm  import WatsonxEmbeddings
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv
import os

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory, BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from pydantic import BaseModel, Field
from typing import Literal

from langchain_community.document_loaders import PyPDFLoader, CSVLoader, WebBaseLoader, DirectoryLoader
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS

from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever, BM25Retriever, SelfQueryRetriever
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

In [10]:
# .env 내용 가져오기
load_dotenv()

apiKey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.environ["HF_TOKEN"]

In [40]:
ollama_embedding = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
watson_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url = f"{watsonx_ai_url}",
    api_key = f"{apiKey}",
    project_id=f"{project_id}"
)

hugging_llm = ChatOpenAI(
  base_url = "https://router.huggingface.co/v1",
  api_key=hf_token,
  model="Qwen/Qwen2.5-7B-Instruct:together",
  temperature=0
)


watson_llm = ChatWatsonx(
  model_id="ibm/granite-4-h-small",
  url=f"{watsonx_ai_url}",
  api_key = f"{apiKey}",
  project_id=f"{project_id}",
  max_tokens = 2000,
  params = {
    "temperature":0
  }
)
qwen_llm = ChatOllama(model="qwen3.5:4b", temperature=0)
exaone_llm = ChatOllama(model="exaone3.5:2.4b", temperature=0)


In [32]:
# pdf -> chunks 함수
def create_chunks_from_pdf(pdf_path, chunk_size=500, chunk_overlap=50):

  loader = PyPDFLoader(pdf_path)
  docs = loader.load()
  splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap)
  chunks = splitter.split_documents(docs)

  # 공백제거
  chunks = [chunk for chunk in chunks if chunk.page_content.strip()]
  return chunks

def create_vectorstore(chunks, embeddings, collection_name, persist_directory="./db/chroma_db"):
  return Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name=collection_name
  )

def create_retriever(vectorstore, search_type="similarity", k=3, fetch_k=20, lambda_mult=0.5):
  kwargs = {'k': k}

  if search_type=="mmr":
    kwargs['fetch_k'] = fetch_k
    kwargs['lambda_mult'] = lambda_mult

  return vectorstore.as_retriever(search_type=search_type, search_kwargs=kwargs)

def print_retrieved_docs(title, retriever, query):
  docs = retriever.invoke(query)

  print("\n"+"="*50)
  print(title)
  print("="*50)

  for i, doc in enumerate(docs):
    print(f"\n[chunk {i}]")
    print(doc.page_content)
    print(f"\nPage: {doc.metadata.get("page")}")

In [ ]:
### 1. 임베딩 모델, 청크 사이즈, 오버랩

In [14]:
chunks1 = create_chunks_from_pdf("./data/Summary of ChatGPTGPT-4 Research.pdf", 1000, chunk_overlap=100)
chunks2 = create_chunks_from_pdf("./data/Summary of ChatGPTGPT-4 Research.pdf", 100, chunk_overlap=10)

print(f"분할된 청크 수: {len(chunks1)}")
print(f"분할된 청크 수: {len(chunks2)}")

분할된 청크 수: 118
분할된 청크 수: 378


In [35]:
vectorstore1 = create_vectorstore(chunks1, watson_embedding, collection_name="gpt_research_watson11")
vectorstore2 = create_vectorstore(chunks2, watson_embedding, collection_name="gpt_research_watson22")

watson1_retriever = create_retriever(vectorstore1)
watson2_retriever = create_retriever(vectorstore2)

query = 'where can i use ChatGPT?'

print_retrieved_docs("chunk=1000,overlap=100", watson1_retriever, query)
print_retrieved_docs("chunk=100,overlap=10", watson2_retriever, query)


chunk=1000,overlap=100

[chunk 0]
development.
2 Related work of ChatGPT
In this section, we review the latest research related to the application, ethics,
and evaluation of ChatGPT.
2.1 Application of ChatGPT
2.1.1 Question And Answering
In the education ﬁeld
ChatGPT is commonly used for question and answers testing in the edu-
cation sector. Users can use ChatGPT to learn, compare and verify answers
for diﬀerent academic subjects such as physics, mathematics, and chemistry,
4

Page: 3

[chunk 1]
While ChatGPT did not perform as well as commercial systems on biomedical
abstracts or Reddit comments, it may be a good speech translator. Prieto et
al. [29] evaluated the use of ChatGPT in developing an automated construction
schedule based on natural language prompts. The experiment required building
new partitions in an existing space and providing details on the rooms to be
partitioned. The results showed that ChatGPT was able to generate a coher-
ent schedule that followed a logical ap

In [37]:
vectorstore3 = create_vectorstore(chunks1, ollama_embedding, collection_name="gpt_research_watson33")
vectorstore4 = create_vectorstore(chunks1, watson_embedding, collection_name="gpt_research_watson44")

watson3_retriever = create_retriever(vectorstore3)
watson4_retriever = create_retriever(vectorstore4)

query = 'where can i use ChatGPT?'

print_retrieved_docs("ollama", watson3_retriever, query)
print_retrieved_docs("watson", watson4_retriever, query)

Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/embeddings?version=2026-05-13)
Status code: 403, body: {"errors":[{"code":"token_quota_reached","message":"Request of 1 token(s) from quota was rejected","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-embeddings"}],"trace":"b1f5abd5719219c8b79347fbe5e7d92e","status_code":403}


ApiRequestFailure: Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/embeddings?version=2026-05-13)
Status code: 403, body: {"errors":[{"code":"token_quota_reached","message":"Request of 1 token(s) from quota was rejected","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-embeddings"}],"trace":"b1f5abd5719219c8b79347fbe5e7d92e","status_code":403}

vectorstore1 = create_vectorstore(chunks1, ollama_embedding, collection_name="gpt_research_watson3")
vectorstore2 = create_vectorstore(chunks2, watson_embedding, collection_name="gpt_research_watson4")

watson1_retriever = create_retriever(vectorstore1)
watson2_retriever = create_retriever(vectorstore2)

query = 'where can i use ChatGPT?'

print_retrieved_docs("chunk=1000,overlap=100", watson1_retriever, query)
print_retrieved_docs("chunk=1000,overlap=100", watson2_retriever, query)

### 2. MMR(Maximal Marginal Relevance) Retriever
- 관련성(Relevance)과 다양성(Diversity) 고려
- 법률 문서, 기술 매뉴얼 처럼 유사 내용이 반복되는 문서에 효과적
- 동작과정
  - 질문 -> 임배딩 -> 백터 스토어에서 질문과 유사한 상위 fetch_k(후보 문서) 추출
  - 후보문서에서 MMR 점수 계산 -> 가장 높은 문서 추출
  - 남은 후보에서 MMR 점수 계산 -> 높은 문서 추출
  - 추출한 높은 문서에서 최종 k 반환

In [34]:
chunks3 = create_chunks_from_pdf("./data/2026 상 삼성전자 DX부문 직무기술서.pdf")
vectorstore1 = create_vectorstore(chunks3, watson_embedding, collection_name="samsung_watson1",persist_directory="./db/watson_chroma2")

mmr_retriever = create_retriever(vectorstore1,search_type="mmr", k=5)
similarity_retriever = create_retriever(vectorstore1, search_type="similarity")

query = '직무 분석'
print_retrieved_docs("MMR", mmr_retriever, query)
print_retrieved_docs("Similarity", similarity_retriever, query)


MMR

[chunk 0]
•
•
•
•
•
•
•

Page: 19

[chunk 1]
•
•
•
•
•
•
•
•
•

Page: 22

[chunk 2]
해외영업
고객과 시장
 제품에 대한 이해를 바탕으로 시장 수요와 경쟁환경을 분석하여 국가
 거래선별 목표 설정
영업전략 수립
 신규 제품
 영업 채널을 발굴하고 판매전략 수립 및 실행을 통해 매출 극대화에
기여합니다

Page: 18

[chunk 3]
S/W개발
소프트웨어 기술에 대한 전문적인 지식을 기반으로 창의적이고 분석적인 사고를 통해 신기술을
선도하고 당사 제품에 반영함으로써 제품 및 솔루션의 혁신적인 가치를 창출합니다

Page: 5

[chunk 4]
•
 ∙
•
•
•
•
•
•
•
•
•
∙

Page: 29

Similarity

[chunk 0]
•
•
•
•
•
•
•

Page: 19

[chunk 1]
•
•
•
•
•
•
•
•
•

Page: 22

[chunk 2]
•
•
•
•
•
•
•
•
•

Page: 27


### 3. SelfQuery Retriever
- 자연어 질문을 분석하여 시멘틱 검색쿼리와 메타데이터필터를 LLM이 자동으로 생성하게 하는 고급 Retriever
- 질문: 2023년 이후 계약금이 1억 이상인 계약 찾아줘 -> LLM
  - 시멘틱 검색쿼리: 계약
  - filter: {year >= 2023, 계약금액 >= 100000}
- 메타데이터 작업이 필요

In [ ]:
docs = [
    Document(
        page_content="삼성전자 제품 마케팅 직무입니다.",
        metadata={
            "year":2025,
            "department":"marketing"
        }
    ),
    Document(
        page_content="AI 연구 개발 직무입니다.",
        metadata={
            "year":2024,
            "department":"ai"
        }
    ),
    Document(
        page_content="백엔드 개발 직무입니다.",
        metadata={
            "year":2025,
            "department":"developer"
        }
    ),
]


metadata_field_info = [
    AttributeInfo(name="year", description="문서 작성 연도", type="integer"),
    AttributeInfo(name="department", description="직무 부서", type="string"),
]

document_content_description = "회사 내부 문서 및 직무 자료"


In [ ]:

vectorstore1 = create_vectorstore(docs, watson_embedding, collection_name="selfquery",persist_directory="./db/watson_chroma")
self_query_retriever = SelfQueryRetriever.from_llm(
  llm=watson_llm,
  vectorstore=vectorstore1,
  metadata_field_info=metadata_field_info,
  verbose=True,
  enable_lint=True,
  structured_query_translator=ChromaTranslator()
)

self_query_retriever.invoke("2024년 이후 ai 부서 직무를 찾아줘")


Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/embeddings?version=2026-05-13)
Status code: 403, body: {"errors":[{"code":"token_quota_reached","message":"Request of 1 token(s) from quota was rejected","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-embeddings"}],"trace":"622403c3552bd70a97b531b28494fdd1","status_code":403}


ApiRequestFailure: Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/embeddings?version=2026-05-13)
Status code: 403, body: {"errors":[{"code":"token_quota_reached","message":"Request of 1 token(s) from quota was rejected","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-embeddings"}],"trace":"622403c3552bd70a97b531b28494fdd1","status_code":403}